# Deep Agents 101: Journal Agent

This notebook builds one agent, one step at a time. Slides cover the concepts (harness, tools, HITL); this notebook covers the implementation.

For topics not covered today, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langchain-openai langgraph tavily-python dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()

# To use an Anthropic/OpenAI/etc. key instead, add api key to .env

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# To use a different key, replace the block above, for example:
# from langchain.chat_models import init_chat_model
# model = init_chat_model("anthropic:claude-haiku-4-5")

## 1: Invoking Your Agent

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model)

Message = "Hello World"

result = agent.invoke({"messages": [{"role": "user", "content": Message}]})
print(result["messages"][-1].content)

## 2: Set its role with a system prompt

In [ ]:
system_prompt = "You are a melodramatic Victorian child. Narrate everything with excessive despair and flowery, dramatic language."

agent = create_deep_agent(model=model, system_prompt=system_prompt)

message = "Log a three-sentence journal entry from these notes: " \
"what I learned today (a caching bug was hiding in the retry logic), " \
"how I felt (relieved but a little anxious)," \
"what's next (write better tests before touching anything else)."

result = agent.invoke({"messages": [{"role": "user", "content": message}]})
print(result["messages"][-1].content)

## 3: Give it a custom tool

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back

In [ ]:
import os
import re
from collections import Counter
from langchain.tools import tool

@tool
def word_count(text: str) -> str:
    """Count the words in a piece of text."""
    return f"{len(text.split())} words"

@tool
def summarize_length(text: str, max_sentences: int = 2) -> str:
    """Trim a piece of text down to its first `max_sentences` sentences."""
    sentences = re.split(r"(?<=[.!?]) +", text.strip())
    return " ".join(sentences[:max_sentences])

In [ ]:
agent = create_deep_agent(model=model, system_prompt=system_prompt, tools=[word_count]) #show adding second tool

result = agent.invoke({"messages": [{"role": "user", "content":
    "Here is a journal entry: 'Today I finally shipped the feature I've been stuck on for a "
    "week. The bug turned out to be a caching issue that took forever to track down, and I "
    "ended up rewriting most of the retry logic to fix it. It feels good to have it done, "
    "though I'm a little worried about whether the fix will hold up under real traffic. "
    "Tomorrow I want to write better tests before touching anything else.' "
    "Count the words in it using your tool, then tell me what you found."
}]})
print(result["messages"][-1].content)

## OPTIONAL: Human-in-the-loop, approve/reject a risky action before it happens

### Bonus: a working example (time permitting)

The cells below build a real version of the pause described above: a `share_journal_entry` tool that requires approval before it runs, using the persona and tool you already picked earlier in this notebook.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def share_journal_entry(entry: str, platform: str) -> str:
    """Share a journal entry to an external platform. This simulates a send, no network call is made."""
    return f"Shared to {platform}: {entry[:60]}..."

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[chosen_tool, share_journal_entry],
    interrupt_on={"share_journal_entry": True},
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-hitl-demo"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Start a journal.md file. Log a new dated entry: 'Set up human-in-the-loop "
        "approval today, it feels reassuring to have a real gate before anything gets "
        "shared externally.' Then read the file back, and share the most recent entry "
        "to the 'team-standup' platform."
    )}]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    print(result["messages"][-1].content)

The cell above paused instead of finishing, because `share_journal_entry` matched `interrupt_on`. The cell below resumes it with an approval decision.

In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(result["messages"][-1].content)

## Wrap-up

You built: a filesystem-backed agent, a way to swap personas, and a custom tool.

Also covered: human-in-the-loop gating, added with a single argument.

Not covered today, but in the full LangChain Academy Deep Agents course: subagent delegation, backends (filesystem/store/composite), skills, memory across sessions, sandboxes, and deployment.